---
**Author:** Leonardo Gabriel Mourao Thiel  
**Project:** Master Thesis – System Inertia in the Energy System of the Future:
Model-Based Cost Optimization to Secure Inertia Requirements

**Notebook:**  Visualization and Statistical Analysis of System Inertia (2024)


**Date:** 27.04.2026  
---

# Visualization and Statistical Analysis of System Inertia (2024)

This notebook presents the visualization and statistical analysis of
system inertia in the European power system for the year 2024.

The analysis is based on precomputed inertia time series, which are
generated in a separate preprocessing notebook. These time series
capture the hourly evolution of system inertia \( H_{sys}(t) \)
for multiple countries.

---

## Background

System inertia is a key indicator of power system stability, as it
determines the system's ability to withstand frequency deviations.

With increasing penetration of renewable energy sources, which typically
do not provide inherent rotational inertia, the overall inertia of the
system may decrease. This raises concerns regarding system stability,
especially during periods of high renewable generation.

---

## Objective

The objective of this notebook is to analyze system inertia from
multiple perspectives:

- cross-country comparison of inertia levels  
- identification of critical low-inertia conditions  
- analysis of temporal patterns (hourly and daily)  
- investigation of statistical properties and distributions  
- assessment of the relationship between generation structure and inertia  

---

## Methodological Approach

The analysis builds on the following steps:

1. Load precomputed system inertia data  
2. Filter the dataset to the relevant analysis period  
3. Compute statistical indicators (mean, quantiles, variability)  
4. Analyze temporal patterns (hourly and daytime aggregation)  
5. Visualize results using multiple plot types  

---

## Key Metrics

The following metrics are used throughout the analysis:

- **Mean and median inertia**  
- **Standard deviation and minimum values**  
- **Share of low-inertia hours** (\( H_{sys} < 2 \, s \))  
- **Temporal averages (hourly and daytime)**  

---

## Scope

- Year: 2024  
- Temporal resolution: hourly  
- Focus period: summer (July–August)  
- Spatial scope: multi-country European system  

---

## Output

The notebook produces:

- statistical summaries  
- comparative visualizations across countries  
- temporal profiles of system inertia  
- indicators of critical system conditions  

These outputs support the interpretation of system stability
and provide the basis for the discussion in the thesis.

---

## Notes

This notebook focuses on analysis and visualization.
The computation of system inertia is performed in a separate notebook.

All results are exported to structured files to ensure reproducibility.

## 1. Environment Setup and Analysis Configuration

This section initializes the analysis environment and defines the
core configuration parameters for the visualization of system inertia.

It includes:

- loading required Python packages  
- defining input and output paths  
- specifying the set of countries  
- selecting the time period for analysis  

The selected time window focuses on the summer period of 2024,
which is typically characterized by high renewable penetration
and potentially lower system inertia.

In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Install required packages (ensures reproducibility)
!{sys.executable} -m pip install -r requirements.txt

# Custom plotting module (contains visualization functions)
import plots_2024 as p


# ---------------------------------------------------------
# Path configuration
# ---------------------------------------------------------

# Base data directory
path = os.path.join("..", "Results")

# Output directory for plots and results
output_path = os.path.join("..", "Results")


# ---------------------------------------------------------
# Model scope: countries included in the analysis
# ---------------------------------------------------------

countryList = [
    "AT","BA","BE","BG","CH","CZ","DE","DK","ES","FR","GR"
]

countryList += [
    "HR","HU","IT","LU","MK","ME","NL","PL","PT","RO","RS","SI","SK"
]


# ---------------------------------------------------------
# Time horizon selection
# ---------------------------------------------------------

# Focus on summer period (high RES share → critical inertia conditions)
start_date = "2024-07-01"
end_date   = "2024-09-01"

ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\Leo\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


## 2. Data Loading and Temporal Filtering

In this section, the precomputed system inertia dataset is loaded
and filtered to the selected analysis period.

The focus is on the summer months (July–August 2024), which are
typically associated with high renewable energy penetration and
potentially lower system inertia.

Restricting the analysis to this period allows for a targeted
investigation of critical system conditions.

In [2]:
# ---------------------------------------------------------
# Load system inertia dataset
# ---------------------------------------------------------

file_path = os.path.join(
    path,
    "2024_inertia",
    "H_sys_hourly_all_countries.xlsx"
)

df = pd.read_excel(file_path)


# ---------------------------------------------------------
# Prepare datetime column
# ---------------------------------------------------------

# Ensure Datetime column is properly parsed
df["Datetime"] = pd.to_datetime(df["Datetime"])


# ---------------------------------------------------------
# Filter to selected time period (summer months)
# ---------------------------------------------------------

mask = (df["Datetime"] >= start_date) & (df["Datetime"] < end_date)
df_filtered = df.loc[mask]


# ---------------------------------------------------------
# Sanity check
# ---------------------------------------------------------

print(df_filtered.head())
print(f"\nNumber of filtered rows: {len(df_filtered)}")

                Datetime        AT        BA        BE        BG        CH  \
4368 2024-07-01 00:00:00  3.605804  4.963580  6.033478  5.495483       NaN   
4369 2024-07-01 01:00:00  3.474139  4.956516  6.234987  5.454139       NaN   
4370 2024-07-01 02:00:00  3.571071  4.939314  6.284769  5.437900  6.456526   
4371 2024-07-01 03:00:00  3.790213  4.962594  6.365247  5.409714  6.460902   
4372 2024-07-01 04:00:00  3.885493  5.012834  6.346214  5.145849  6.576315   

            CZ        DE        DK        ES  ...        LU        MK  \
4368  5.637844  4.450849  2.543409  4.501022  ...  2.689409  5.746436   
4369  5.648540  4.421348  2.528532  4.385796  ...  3.965294  5.712122   
4370  5.657948  4.554011  2.500506  4.251755  ...  5.135024  5.734687   
4371  5.596904  4.576392  2.727037  4.160596  ...  5.202086  5.738986   
4372  5.585335  4.555589  2.968060  4.139390  ...  4.329837  5.747899   

            ME        NL        PL        PT        RO        RS        SI  \
4368  4.353977

## 3. Average System Inertia per Country

In this section, the average system inertia is computed for each country
over the selected time period.

The mean value of \( H_{sys}(t) \) provides a first aggregated indicator
of the typical inertia level in each country during the summer period.

---

### Purpose

- compare overall inertia levels across countries  
- identify regions with structurally low inertia  
- provide a baseline for further analysis  

---

### Interpretation

- **Higher average inertia** → generally more stable system conditions  
- **Lower average inertia** → potentially higher vulnerability to disturbances  

In [3]:
# ---------------------------------------------------------
# Identify country columns (exclude datetime)
# ---------------------------------------------------------

country_columns = df_filtered.columns.drop("Datetime")


# ---------------------------------------------------------
# Compute average inertia per country
# ---------------------------------------------------------

average_per_country = df_filtered[country_columns].mean()


# ---------------------------------------------------------
# Output results
# ---------------------------------------------------------

print("Average system inertia per country:")
print(average_per_country.sort_values(ascending=False))

Average system inertia per country:
MK    5.903375
SK    5.865672
NL    5.808732
CH    5.696044
FR    5.272449
IT    5.199941
CZ    5.177643
HR    5.079679
SI    5.050446
BA    4.973926
RO    4.964276
HU    4.856142
BE    4.834529
RS    4.708312
ME    4.669533
BG    4.617545
PL    4.320041
GR    4.065822
AT    4.036708
ES    3.761606
LU    3.398462
DE    3.331093
PT    3.131266
DK    2.108560
dtype: float64


Erzeugungsdaten Gesamtnetz laden udn Spaltennamen aufbereiten für Match

## 4. Statistical Analysis of System Inertia

This section provides a statistical characterization of system inertia
across countries during the selected time period.

Multiple statistical indicators are computed to capture both
central tendencies and extreme conditions.

---

### Metrics

The following metrics are evaluated for each country:

- **mean**: average inertia level  
- **median**: robust central tendency  
- **standard deviation**: variability of inertia  
- **minimum**: lowest observed inertia (critical condition)  
- **share < 2 s**: percentage of hours with low inertia  
- **quantilles**: quantilles of [0.1, 0.25, 0.75, 0.9] 


---

### Motivation

While average values provide a general overview, extreme values and
low-inertia events are particularly important, as they indicate
potentially critical system states.

In [4]:
import pandas as pd
import os

# ---------------------------------------------------------
# Prepare data
# ---------------------------------------------------------

# Identify country columns (exclude datetime)
country_columns = df_filtered.columns.drop("Datetime")

# Ensure numeric values (convert invalid entries to NaN)
df_filtered[country_columns] = df_filtered[country_columns].apply(
    pd.to_numeric, errors="coerce"
)


# ---------------------------------------------------------
# 1) Summary statistics
# ---------------------------------------------------------

summary = pd.DataFrame({
    "mean": df_filtered[country_columns].mean(),
    "median": df_filtered[country_columns].median(),
    "std": df_filtered[country_columns].std(),
    "min": df_filtered[country_columns].min(),

    # Share of hours with very low inertia (< 2 s)
    "share_<2_%": (df_filtered[country_columns] < 2).mean() * 100
})


# ---------------------------------------------------------
# 2) Quantiles
# ---------------------------------------------------------

quantiles = df_filtered[country_columns].quantile(
    [0.10, 0.25, 0.50, 0.75, 0.90]
).rename(index={
    0.10: "q10",
    0.25: "q25",
    0.50: "median",
    0.75: "q75",
    0.90: "q90"
})

quantiles = df_filtered[country_columns].quantile([0.1, 0.25, 0.75, 0.9])

# ---------------------------------------------------------
# 3) Count of critical hours
# ---------------------------------------------------------

counts_below_2 = (df_filtered[country_columns] < 2).sum().to_frame(
    name="count_<2"
)




## 5. Temporal Analysis of System Inertia

This section analyzes the temporal patterns of system inertia
over the course of a typical day.

The objective is to identify systematic variations in inertia
levels across different hours and times of day.

---

### Analysis Components

Two complementary perspectives are considered:

1. **Hourly averages (0–23)**  
   - capture fine-grained daily patterns  

2. **Aggregated time-of-day categories**  
   - night (0–5)  
   - morning (6–11)  
   - afternoon (12–17)  
   - evening (18–23)  

---

### Motivation

System inertia is expected to vary with generation patterns:

- lower inertia during periods of high renewable penetration  
- higher inertia during periods with more conventional generation  

---

### Key Metric

In addition to mean inertia values, the share of low-inertia hours
(\( H_{sys} < 2 \, s \)) is evaluated to identify critical time periods.

In [5]:
# ---------------------------------------------------------
# Extract hour from datetime
# ---------------------------------------------------------

df_filtered["hour"] = df_filtered["Datetime"].dt.hour


# ---------------------------------------------------------
# 1) Hourly average inertia (0–23)
# ---------------------------------------------------------

hourly_mean = (
    df_filtered
    .groupby("hour")[country_columns]
    .mean()
    .reindex(range(24))  # ensure all hours are present
)


# ---------------------------------------------------------
# 2) Define time-of-day categories
# ---------------------------------------------------------

def hour_to_daytime(hour):
    if 0 <= hour <= 5:
        return "Nacht"
    elif 6 <= hour <= 11:
        return "Morgen"
    elif 12 <= hour <= 17:
        return "Nachmittag"
    else:
        return "Abend"


df_filtered["Tageszeit"] = df_filtered["hour"].apply(hour_to_daytime)


# ---------------------------------------------------------
# 3) Average inertia per time of day
# ---------------------------------------------------------

daytime_mean = (
    df_filtered
    .groupby("Tageszeit")[country_columns]
    .mean()
    .reindex(["Nacht", "Morgen", "Nachmittag", "Abend"])
)


# ---------------------------------------------------------
# 4) Share of low-inertia hours (< 2 s)
# ---------------------------------------------------------

daytime_share_below_2 = (
    (df_filtered[country_columns] < 2)
    .groupby(df_filtered["Tageszeit"])
    .mean() * 100
).reindex(["Nacht", "Morgen", "Nachmittag", "Abend"])





## 6. Export of Statistical Results

This section exports all computed statistical indicators and temporal
aggregations to a structured Excel file.

The exported sheets provide a comprehensive overview of system inertia,
including:

- central statistics  
- extreme values and critical events  
- temporal patterns (hourly and daytime)  

This dataset serves as the basis for further interpretation
and documentation in the thesis.

In [6]:
# ---------------------------------------------------------
# 1) Summary statistics
# ---------------------------------------------------------

summary = pd.DataFrame({
    "mean": df_filtered[country_columns].mean(),
    "median": df_filtered[country_columns].median(),
    "std": df_filtered[country_columns].std(),
    "min": df_filtered[country_columns].min(),
    "share_<2_%": (df_filtered[country_columns] < 2).mean() * 100
})

# Sortierte Version für bessere Lesbarkeit
summary_sorted = summary.sort_values("mean")


# ---------------------------------------------------------
# Output configuration
# ---------------------------------------------------------

output_folder = os.path.join(output_path, "2024_inertia")
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(
    output_folder,
    "H_sys_summary_2024.xlsx"
)


# ---------------------------------------------------------
# Export to Excel
# ---------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    # Core statistics
    summary.to_excel(writer, sheet_name="summary")
    summary_sorted.to_excel(writer, sheet_name="summary_sorted")

    # Critical conditions
    counts_below_2.to_excel(writer, sheet_name="counts_below_2")

    # Temporal patterns
    daytime_mean.to_excel(writer, sheet_name="daytime_mean")
    daytime_share_below_2.to_excel(writer, sheet_name="share_below_2")
    hourly_mean.to_excel(writer, sheet_name="hourly_mean")

print("✅ Results exported successfully:", output_file)

✅ Results exported successfully: ..\Results\2024_inertia\H_sys_summary_2024.xlsx


## 7. Visualization of System Inertia

This section presents a set of visualizations to analyze and interpret
system inertia across countries and time.

The plots are designed to highlight:

- cross-country differences  
- temporal patterns  
- distribution characteristics  
- critical low-inertia conditions  

---

### Visualization Categories

The analysis includes:

1. **Country comparison**
   - median inertia levels  
   - share of low-inertia hours  

2. **Temporal patterns**
   - hourly profiles  
   - daytime aggregation  

3. **Distribution analysis**
   - boxplots per country  

4. **System structure**
   - comparison of conventional vs. non-conventional generation  

---

### Interpretation Focus

The visualizations aim to identify:

- countries with structurally low inertia  
- time periods with increased system vulnerability  
- relationships between generation mix and inertia levels  

In [7]:
# ---------------------------------------------------------
# 1) Cross-country comparison
# ---------------------------------------------------------

# Median inertia per country
p.median_plot(summary_sorted)

# Share of low-inertia hours (< 2 s)
p.share_below_2(summary_sorted)


# ---------------------------------------------------------
# 2) Temporal analysis
# ---------------------------------------------------------

# Hourly average inertia profile
p.hourly_mean(hourly_mean)

# Daytime aggregation of low-inertia share
p.daytime_share_below_2(summary_sorted, daytime_share_below_2)

# Detailed hourly profiles (line plots)
p.hourly_profiles_lines(hourly_mean)


# ---------------------------------------------------------
# 3) Distribution analysis
# ---------------------------------------------------------

# Distribution of inertia values per country
p.boxplot_per_country(df_filtered, country_columns)


# ---------------------------------------------------------
# 4) System structure
# ---------------------------------------------------------

# Compare conventional vs. non-conventional generation shares
p.generation_share_konventionell_vs_nicht_konventionell(
    countryList,
    start_date,
    end_date
)

Saved: ../Results/2024\mean_median_errorbars_per_country.pdf
Saved: ../Results/2024\mean_median_errorbars_per_country.png
Saved: ../Results/2024\share_below_2_per_country.pdf
Saved: ../Results/2024\share_below_2_per_country.png
Saved: ../Results/2024\hourly_mean_by_country.pdf
Saved: ../Results/2024\hourly_mean_by_country.png
Saved: ../Results/2024\daytime_share_below_2.pdf
Saved: ../Results/2024\daytime_share_below_2.png
Saved: ../Results/2024\hourly_profiles_lines.pdf
Saved: ../Results/2024\hourly_profiles_lines.png
Saved: ../Results/2024\boxplot_per_country.pdf
Saved: ../Results/2024\boxplot_per_country.png
Saved: ../Results/2024\generation_share_konventionell_vs_nicht_konventionell.pdf
Saved: ../Results/2024\generation_share_konventionell_vs_nicht_konventionell.png
